# PKG Attrition — Source Profiling EDA · v2

**Rewritten against the confirmed schema.** No column resolution — names are known.

### What changed from v1, and why it matters

| v1 assumed | Reality | Consequence |
|---|---|---|
| a `direction` column | **direction is the null pattern** on `mdm_id_pays` / `mdm_id_receives` | topology classification replaces sign-convention checking (Stage 2) |
| RTN + account to be built | `unq_cpty_acct_id` is **pre-built upstream** | key provenance is the question, not key construction (Stage 4) |
| an FI list to be assembled | `cpty_fin_entity_name` **already names the bank** | `fi_destination_flag` is a lookup, not a matching problem (Stage 5) |
| long deposit panel, account grain | **wide, customer grain, monthly average** | unpivot required; no account roll-up needed (Stage 8) |
| strict ledger reconciliation | monthly *average* balance | exact reconciliation is impossible; measure explained variance instead (Stage 10) |
| status column might exist | **it does not** | Stage 14 builds candidate labels from what is here (§ pinned request) |

### The three findings that decide the study

1. **Stage 4** — counterparty key provenance and name coverage. Decides whether Group A exists.
2. **Stage 10/11** — whether staging explains balance movement, and whether account-level
   activity can stand in for the missing monthly closed-account count.
3. **Stage 13** — graph-visible episode count. Decides whether Steps 3–6 of the brief are powered.

### Pinned for the deposit team
`latest_num_closed_accounts` is current-state only. A **monthly** closed-account count, or
per-account open/close dates, converts this study from "predict a balance-derived event"
to "predict an actual departure" and removes the circularity that handicapped the last attempt.

In [ ]:
# ============================================================================
# CONFIG
# ============================================================================
import os, re, json
import pandas as pd
from pyspark.sql import SparkSession, functions as F, Window as W

CONFIG = dict(
    TXN_TABLE  = "<db>.<staging_transactions>",
    DEP_TABLE  = "<db>.<deposit_wide>",
    CUST_TABLE = "<db>.neo4j_customer",

    OUT_DIR    = "../eda/attrition",

    # Scan window. Widen once Stage 1 confirms the true range.
    DATE_MIN   = "2025-01-01",
    DATE_MAX   = "2025-12-31",

    # Deposit panel covers 2024-01 .. 2026-06 in the wide layout.
    BAL_PREFIX = "bal_",

    # §6.2 of the brief
    BALANCE_FLOOR     = 25_000,
    CURRENT_RULE_DROP = 0.30,

    # Stage 14: months of zero transaction activity that constitute "silence"
    SILENCE_MONTHS    = 3,
)

# ---- staging table columns (confirmed) --------------------------------------
T = dict(
    pays_id     = "mdm_id_pays",
    pays_name   = "customer_name_pays",
    recv_id     = "mdm_id_receives",
    recv_name   = "customer_name_receives",
    pays_acct   = "pnc_dep_acct_pays",
    recv_acct   = "pnc_dep_acct_receives",
    cpty_key    = "unq_cpty_acct_id",
    cpty_name   = "cpty_name",
    cpty_type   = "cpty_type",
    cpty_fi     = "cpty_fin_entity_name",
    txn_id      = "trans_id",
    amount      = "trans_amt",
    currency    = "trans_currency",
    purpose     = "trans_purpose",
    date        = "trans_dt",
    rail        = "payment_rail",
    category    = "category",
    cat_prefix  = "category_prefix",
    src_syst    = "src_syst",
    np_key      = "np_key",
    merch_id    = "merchant_id",
    merch_cat   = "merchant_cat_cd",
    merch_state = "merchant_state",
    load_ts     = "hdfs_load_ts",
)

# ---- deposit table columns (confirmed) --------------------------------------
DEP = dict(
    cust        = "cust_pwr_id",
    rltn        = "rltn_pwr_id",
    latest      = "latest_month",
    prior_avg   = "prior_avg",
    recent_avg  = "recent_avg",
    pct_change  = "pct_change",
    peak        = "peak_balance",
    latest_bal  = "latest_month_bal",
    n_acct      = "latest_num_accounts",
    n_closed    = "latest_num_closed_accounts",
)

# ---- customer dimension ------------------------------------------------------
CUST = dict(
    mdm_id   = "mdm_id",
    pwr_id   = "cust_pwr_id",
    party    = "party_type",     # P / O / null
    naics    = "naics_cd",
    name     = "customer_name",
)

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "400")

OUT = CONFIG["OUT_DIR"]
os.makedirs(OUT, exist_ok=True)
R = {}   # headline results -> Stage 15


def save(pdf, name, note=""):
    path = os.path.join(OUT, f"{name}.csv")
    pdf.to_csv(path, index=False)
    print(f"\n--- {name}{' · ' + note if note else ''}")
    with pd.option_context("display.max_rows", 80, "display.width", 220,
                           "display.float_format", lambda v: f"{v:,.4f}"):
        print(pdf.to_string(index=False))
    print(f"    -> {path}")
    return pdf


def norm_name(col):
    """Uppercase, drop punctuation and legal suffixes, collapse whitespace.

    Deliberately exact, not fuzzy. Fuzzy expansion is a later layer that gets
    calibrated against the false-positive estimate in Stage 12 — it does not get
    switched on because it sounds better.
    """
    c = F.upper(F.trim(col))
    c = F.regexp_replace(c, r"[^A-Z0-9 ]", " ")
    c = F.regexp_replace(c, r"\b(LLC|L L C|INC|INCORPORATED|CORP|CORPORATION|CO|LP|LLP|"
                            r"PLLC|PC|LTD|LIMITED|DBA|THE)\b", " ")
    return F.trim(F.regexp_replace(c, r"\s+", " "))


def present(c):
    """Non-null and non-blank. Parquet keeps NULL and '' distinct; every test here
    asks only 'is there a value', so the two are collapsed once, in one place."""
    return F.col(c).isNotNull() & (F.trim(F.col(c).cast("string")) != "")


def dollar_share(df, group_cols, name, note="", amt=None):
    amt = amt or T["amount"]
    g = (df.groupBy(*group_cols)
           .agg(F.count(F.lit(1)).alias("n_rows"),
                F.sum(F.col(amt).cast("double")).alias("dollars"))
           .toPandas())
    g["share_rows"] = g["n_rows"] / g["n_rows"].sum()
    g["share_dollars"] = g["dollars"] / g["dollars"].sum()
    return save(g.sort_values("dollars", ascending=False), name, note)


print("config loaded ·", OUT)

---
## Stage 1 — Staging shape, grain, and coverage

Row count, true date range, and whether `trans_id` is unique. If it is not, an
internal customer-to-customer payment is present twice — once on each side's
ledger — and every count below is a leg count. That has to be known before any
aggregate is quoted, and it is silent if unchecked.

In [ ]:
txn_all = spark.table(CONFIG["TXN_TABLE"])

shape = txn_all.agg(
    F.count(F.lit(1)).alias("n_rows"),
    F.countDistinct(T["txn_id"]).alias("n_trans_ids"),
    F.min(T["date"]).alias("min_dt"),
    F.max(T["date"]).alias("max_dt"),
    F.sum(F.col(T["amount"]).cast("double")).alias("dollars"),
    F.max(T["load_ts"]).alias("max_load_ts"),
).toPandas()
shape["rows_per_trans_id"] = shape["n_rows"] / shape["n_trans_ids"]
save(shape.T.reset_index().rename(columns={"index": "metric", 0: "value"}),
     "01_staging_shape",
     "rows_per_trans_id > 1 => internal payments are double-booked; treat as legs")
R["rows_per_trans_id"] = float(shape["rows_per_trans_id"].iloc[0])

In [ ]:
# Monthly volume against the deposit panel's own range (2024-01 .. 2026-06).
# A staging table that stops early recreates the freeze structure by accident,
# which is exactly the confound the last study had to be engineered around.
monthly = (txn_all
    .withColumn("month", F.substring(F.col(T["date"]), 1, 7))
    .groupBy("month")
    .agg(F.count(F.lit(1)).alias("n_rows"),
         F.sum(F.col(T["amount"]).cast("double")).alias("dollars"),
         F.countDistinct(T["pays_id"]).alias("n_payers"),
         F.countDistinct(T["recv_id"]).alias("n_receivers"))
    .orderBy("month").toPandas())
save(monthly, "01_monthly_volume",
     "compare the tail against deposit coverage through 2026-06")

In [ ]:
# Currency and amount sanity. Non-USD rows and negative amounts (reversals,
# returns) both need an explicit policy before anything is summed.
save(txn_all.groupBy(T["currency"]).agg(
        F.count(F.lit(1)).alias("n"),
        F.sum(F.col(T["amount"]).cast("double")).alias("dollars")).toPandas(),
     "01_currency_mix")

save(txn_all.agg(
        F.sum(F.when(F.col(T["amount"]) < 0, 1).otherwise(0)).alias("n_negative"),
        F.sum(F.when(F.col(T["amount"]) == 0, 1).otherwise(0)).alias("n_zero"),
        F.expr(f"percentile_approx(cast({T['amount']} as double), 0.5)").alias("p50"),
        F.expr(f"percentile_approx(cast({T['amount']} as double), 0.999)").alias("p999"),
        F.max(F.col(T["amount"]).cast("double")).alias("max"),
     ).toPandas(),
     "01_amount_sanity", "negatives are reversals/returns and are not outflow")

In [ ]:
txn = (txn_all
       .filter((F.col(T["date"]) >= CONFIG["DATE_MIN"]) & (F.col(T["date"]) <= CONFIG["DATE_MAX"]))
       .cache())
print("scan window rows:", txn.count())

---
## Stage 2 — Topology: direction from the null pattern

There is no direction column. Direction *is* the null pattern:

| `mdm_id_pays` | `mdm_id_receives` | class | meaning |
|---|---|---|---|
| present | present | `INTERNAL_C2C` | both sides are PNC customers — the existing PKG edge |
| absent | present | `INBOUND` | external counterparty pays a customer |
| present | absent | `OUTBOUND` | customer pays an external counterparty |
| absent | absent | `ORPHAN` | neither side identified — should be ~0, and is a data defect if not |

**`OUTBOUND` is the half the graph has never had.** Everything the brief calls
Group A lives here. `INTERNAL_C2C` is the ~12%-coverage world the prior study
was confined to.

Self-loops (`pays_id == receives_id`) are one customer moving money between its
own PNC accounts. They are not payments and must not enter flow metrics — but
they are the only on-us analogue of the off-us self-payment we want to detect,
so they are counted rather than silently dropped.

In [ ]:
txn = (txn
    .withColumn("_pays", present(T["pays_id"]))
    .withColumn("_recv", present(T["recv_id"]))
    .withColumn("topology",
        F.when(F.col("_pays") & F.col("_recv"), "INTERNAL_C2C")
         .when(~F.col("_pays") & F.col("_recv"), "INBOUND")
         .when(F.col("_pays") & ~F.col("_recv"), "OUTBOUND")
         .otherwise("ORPHAN"))
    .withColumn("is_self_loop",
        F.col("_pays") & F.col("_recv") & (F.col(T["pays_id"]) == F.col(T["recv_id"])))
    .cache())

dollar_share(txn, ["topology"], "02_topology_mix",
             "*** OUTBOUND is the new half. ORPHAN > ~0 is a defect. ***")
dollar_share(txn, ["topology", T["rail"]], "02_topology_x_rail",
             "which rails carry the outbound view")

save(txn.filter("is_self_loop").agg(
        F.count(F.lit(1)).alias("n_self_loops"),
        F.countDistinct(T["pays_id"]).alias("n_customers"),
        F.sum(F.col(T["amount"]).cast("double")).alias("dollars")).toPandas(),
     "02_self_loops",
     "own-account transfers: exclude from flow, keep as the on-us self-payment analogue")

R["outbound_dollar_share"] = float(
    txn.filter("topology='OUTBOUND'").agg(F.sum(F.col(T["amount"]).cast("double"))).first()[0] or 0
) / float(txn.agg(F.sum(F.col(T["amount"]).cast("double"))).first()[0] or 1)

In [ ]:
# The PNC-side account columns. Deposits are customer-grain so these cannot join
# there — but they give a per-month count of ACTIVE accounts, which is the closest
# available proxy for the monthly closed-account count that does not exist (Stage 11).
save(txn.select(
        F.mean(present(T["pays_acct"]).cast("double")).alias("pays_acct_populated"),
        F.mean(present(T["recv_acct"]).cast("double")).alias("recv_acct_populated"),
     ).toPandas(),
     "02_pnc_account_population", "gates the Stage 11 account-activity proxy")

---
## Stage 3 — Rail and product taxonomy

`payment_rail`, `category`, `category_prefix`, `src_syst` and `np_key` are five
overlapping descriptors. The rail metrics spec has been blocked on knowing which
is the real taxonomy and how internal book transfers are represented. This stage
settles it by cross-tabbing them rather than by picking one on faith.

In [ ]:
for c, nm in [(T["rail"], "03_rail"), (T["category"], "03_category"),
              (T["cat_prefix"], "03_category_prefix"), (T["src_syst"], "03_src_syst")]:
    dollar_share(txn, [c], nm)

# Cardinality of np_key decides whether it is a taxonomy or an identifier.
save(txn.agg(F.countDistinct(T["np_key"]).alias("n_np_key"),
             F.countDistinct(T["rail"]).alias("n_rail"),
             F.countDistinct(T["category"]).alias("n_category")).toPandas(),
     "03_taxonomy_cardinality", "high np_key cardinality => identifier, not a class")

dollar_share(txn, [T["rail"], T["category"]], "03_rail_x_category",
             "is category a refinement of rail, or an orthogonal axis?")

In [ ]:
# Amount distribution per rail. Card sits orders of magnitude below wire; a single
# dollar threshold across rails is meaningless, and every "large outflow" rule in
# the brief needs to be rail-relative.
save(txn.groupBy(T["rail"]).agg(
        F.count(F.lit(1)).alias("n"),
        F.expr(f"percentile_approx(cast({T['amount']} as double), 0.5)").alias("p50"),
        F.expr(f"percentile_approx(cast({T['amount']} as double), 0.95)").alias("p95"),
        F.expr(f"percentile_approx(cast({T['amount']} as double), 0.999)").alias("p999"),
     ).toPandas(),
     "03_amount_by_rail")

---
## Stage 4 — Counterparty identifiability  *(the Group A gate)*

`unq_cpty_acct_id` is built upstream as `rn + account_id`, falling back to the
**name** when the account is absent. Those two are not the same object:

- **account-derived** → a stable node. The same counterparty account keys
  identically every month, and appearing / disappearing is a real event.
- **name-derived** → a node whose identity is a string. Two spellings are two
  nodes; two firms with one name are one node. Relationship-dissolution metrics
  (Group B) are unreliable on these, because a spelling change reads as a lost
  relationship.

The provenance split is therefore the single most important number in this stage,
and it is inferable from the key's own format.

In [ ]:
ext = txn.filter("topology IN ('INBOUND','OUTBOUND')").cache()

ext = (ext
    .withColumn("_has_key", present(T["cpty_key"]))
    .withColumn("_has_name", present(T["cpty_name"]))
    .withColumn("_has_fi", present(T["cpty_fi"]))
    # Heuristic: an rn+account concatenation is all digits (allow separators).
    # A name fallback carries alphabetic characters. Verified by the length and
    # sample listings below — do not trust the flag without reading them.
    .withColumn("key_provenance",
        F.when(~F.col("_has_key"), "none")
         .when(F.col(T["cpty_key"]).rlike(r"^[0-9][0-9\-_|]*[0-9]$"), "account_derived")
         .otherwise("name_derived")))

dollar_share(ext, ["key_provenance"], "04_key_provenance",
             "*** account_derived is a stable node; name_derived is a string ***")
dollar_share(ext, ["topology", "key_provenance"], "04_key_provenance_by_topology")
dollar_share(ext, [T["rail"], "key_provenance"], "04_key_provenance_by_rail",
             "name-derived keys will concentrate in specific rails")

# Verify the heuristic by eye before believing any of the above.
save(ext.select(T["cpty_key"], "key_provenance", T["rail"])
        .filter(present(T["cpty_key"])).limit(40).toPandas(),
     "04_key_format_sample", "*** READ THIS — confirm the provenance regex is right ***")

save(ext.filter(present(T["cpty_key"]))
        .select(F.length(T["cpty_key"]).alias("key_len"), "key_provenance")
        .groupBy("key_provenance", "key_len").count()
        .orderBy("key_provenance", "key_len").limit(60).toPandas(),
     "04_key_length_distribution")

In [ ]:
# Joint identifiability. This is the architecture decision: if key + name + FI is
# dominant, Group A is fully computable and the fuzzy matcher is optional.
ext = ext.withColumn("id_class",
    F.when(F.col("_has_key") & F.col("_has_name") & F.col("_has_fi"), "key+name+fi")
     .when(F.col("_has_key") & F.col("_has_name"), "key+name")
     .when(F.col("_has_key"), "key_only")
     .when(F.col("_has_name"), "name_only")
     .otherwise("unidentifiable"))

dollar_share(ext, ["id_class"], "04_identifiability_classes",
             "*** THE Q3 ANSWER, dollar-weighted ***")
dollar_share(ext, [T["rail"], "id_class"], "04_identifiability_by_rail",
             "coverage is rail-biased; rail mix correlates with industry and size")

R["cpty_name_dollar_coverage"] = float(
    ext.agg((F.sum(F.when(F.col("_has_name"), F.col(T["amount"]).cast("double")).otherwise(0.0))
             / F.sum(F.col(T["amount"]).cast("double")))).first()[0] or 0)
R["cpty_key_dollar_coverage"] = float(
    ext.agg((F.sum(F.when(F.col("_has_key"), F.col(T["amount"]).cast("double")).otherwise(0.0))
             / F.sum(F.col(T["amount"]).cast("double")))).first()[0] or 0)

In [ ]:
# cpty_type — if this already separates individual / business / FI, it saves the
# entity-typing work entirely on the counterparty side.
dollar_share(ext, [T["cpty_type"]], "04_cpty_type_mix",
             "does this already carry entity typing?")
dollar_share(ext, [T["cpty_type"], "key_provenance"], "04_cpty_type_x_provenance")

In [ ]:
# Fan-out and fan-in. Fan-out sizes the Tier-1 monthly edge table. Fan-in decides
# whether Group C is computable: a counterparty seen by ONE customer has no
# independent health signal, so lost_cpty_health_index is undefined for it.
keyed = ext.filter(present(T["cpty_key"])).withColumn("month", F.substring(T["date"], 1, 7))

fan_out = (keyed.filter("topology='OUTBOUND'")
    .groupBy(T["pays_id"], "month").agg(F.countDistinct(T["cpty_key"]).alias("n_cpty"))
    .agg(F.count(F.lit(1)).alias("n_customer_months"),
         F.mean("n_cpty").alias("mean"),
         F.expr("percentile_approx(n_cpty, 0.5)").alias("p50"),
         F.expr("percentile_approx(n_cpty, 0.95)").alias("p95"),
         F.expr("percentile_approx(n_cpty, 0.999)").alias("p999"),
         F.max("n_cpty").alias("max"),
         F.sum("n_cpty").alias("total_edges")).toPandas())
save(fan_out, "04_outbound_fanout", "total_edges sizes the Tier-1 monthly graph")

fan_in = (keyed.groupBy(T["cpty_key"])
    .agg(F.countDistinct(F.coalesce(F.col(T["pays_id"]), F.col(T["recv_id"]))).alias("n_customers")))
save(fan_in.agg(
        F.count(F.lit(1)).alias("n_counterparties"),
        F.mean((F.col("n_customers") >= 2).cast("double")).alias("share_shared_2plus"),
        F.mean((F.col("n_customers") >= 5).cast("double")).alias("share_shared_5plus"),
        F.max("n_customers").alias("max_customers")).toPandas(),
     "04_cpty_fanin", "*** share_shared_5plus bounds Group C coverage ***")

---
## Stage 5 — `cpty_fin_entity_name`: the FI registry, already built

This column removes the whole §12 open item. `fi_destination_flag` becomes a
lookup and "which bank are they moving to" is answerable by name.

Three questions: how well is it populated, whether PNC appears in it (those rows
are on-us and must not count as off-us flow), and what the destination
distribution looks like — because the top of that list *is* the competitive map.

In [ ]:
save(ext.groupBy("topology").agg(
        F.mean(F.col("_has_fi").cast("double")).alias("fi_populated_rows"),
        (F.sum(F.when(F.col("_has_fi"), F.col(T["amount"]).cast("double")).otherwise(0.0))
         / F.sum(F.col(T["amount"]).cast("double"))).alias("fi_populated_dollars")).toPandas(),
     "05_fi_coverage", "*** this replaces the FI-name-list open item entirely ***")

top_fi = (ext.filter(F.col("_has_fi"))
    .groupBy(T["cpty_fi"])
    .agg(F.count(F.lit(1)).alias("n"),
         F.countDistinct(F.coalesce(F.col(T["pays_id"]), F.col(T["recv_id"]))).alias("n_customers"),
         F.sum(F.col(T["amount"]).cast("double")).alias("dollars"))
    .orderBy(F.desc("dollars")).limit(100).toPandas())
save(top_fi, "05_top_financial_entities",
     "*** the competitive map. Check whether PNC itself appears — those are on-us. ***")

# Outbound only, which is the direction that matters for attrition.
dollar_share(ext.filter("topology='OUTBOUND' AND _has_fi"), [T["cpty_fi"]],
             "05_outbound_destination_banks", "where money leaves to")

In [ ]:
# Top counterparty names — the hub seed. Payroll processors, card networks and
# PNC's own book-transfer accounts should be visible here, and each needs a
# DIFFERENT policy: payroll is signal, card network is noise, book transfer is
# internal. A flat exclusion list collapses all three.
save(ext.filter(F.col("_has_name"))
        .groupBy(norm_name(F.col(T["cpty_name"])).alias("norm_name"))
        .agg(F.count(F.lit(1)).alias("n"),
             F.countDistinct(F.coalesce(F.col(T["pays_id"]), F.col(T["recv_id"]))).alias("n_customers"),
             F.sum(F.col(T["amount"]).cast("double")).alias("dollars"),
             F.first(T["cpty_fi"], ignorenulls=True).alias("fi_sample"),
             F.first(T["cpty_type"], ignorenulls=True).alias("type_sample"))
        .orderBy(F.desc("n_customers")).limit(100).toPandas(),
     "05_top_counterparty_names", "hub taxonomy seed: payroll / network / internal / anchor")

---
## Stage 6 — The merchant namespace

Card transactions carry `merchant_*` rather than counterparty fields. That is a
**second counterparty namespace**, and whether it also populates
`unq_cpty_acct_id` determines whether card flow joins the same graph or sits
beside it. Either answer is workable; not knowing which produces a graph where
some card spend is an edge and some is not.

In [ ]:
merch = txn.withColumn("_has_merch", present(T["merch_id"]))

save(merch.withColumn("_has_cpty", present(T["cpty_key"]))
        .groupBy("_has_merch", "_has_cpty")
        .agg(F.count(F.lit(1)).alias("n"),
             F.sum(F.col(T["amount"]).cast("double")).alias("dollars")).toPandas(),
     "06_merchant_vs_cpty_population",
     "*** merchant-only rows are a separate namespace — decide include/exclude explicitly ***")

dollar_share(txn.filter(present(T["merch_id"])), [T["rail"]], "06_merchant_rails")

---
## Stage 7 — Customer linkage and the corporate population

`mdm_id` ↔ `cust_pwr_id` is stated one-to-one. Verified rather than assumed,
because if it fans out, one "customer" in the deposit table is several graph
nodes and every per-customer aggregate is wrong by an unknown factor.

`party_type` is then the population gate. At ~82% individuals in the graph,
any aggregate that does not partition on it is substantially a retail statement.

In [ ]:
cust = spark.table(CONFIG["CUST_TABLE"])

link = cust.agg(
    F.count(F.lit(1)).alias("n_rows"),
    F.countDistinct(CUST["mdm_id"]).alias("n_mdm_id"),
    F.countDistinct(CUST["pwr_id"]).alias("n_cust_pwr_id"),
    F.sum(F.when(present(CUST["pwr_id"]), 0).otherwise(1)).alias("n_null_pwr_id"),
).toPandas()
save(link.T.reset_index().rename(columns={"index": "metric", 0: "value"}),
     "07_customer_link_shape", "n_mdm_id == n_cust_pwr_id confirms 1:1")

# Explicit fanout check in both directions.
save(cust.filter(present(CUST["pwr_id"]))
        .groupBy(CUST["pwr_id"]).agg(F.countDistinct(CUST["mdm_id"]).alias("n_mdm"))
        .groupBy("n_mdm").count().orderBy("n_mdm").limit(20).toPandas(),
     "07_mdm_per_pwr_id", "*** anything above n_mdm=1 breaks the 1:1 assumption ***")

save(cust.groupBy(CUST["party"]).agg(
        F.count(F.lit(1)).alias("n"),
        F.mean(present(CUST["pwr_id"]).cast("double")).alias("pwr_id_populated")).toPandas(),
     "07_party_type_mix", "the population gate — O is the TM population")

In [ ]:
# Bind the mapping used everywhere below. Organisations only.
m2p = (cust.filter(present(CUST["pwr_id"]))
           .select(F.col(CUST["mdm_id"]).alias("mdm_id"),
                   F.col(CUST["pwr_id"]).alias("cust_pwr_id"),
                   F.col(CUST["party"]).alias("party_type"),
                   F.col(CUST["naics"]).alias("naics_cd"))
           .distinct().cache())
print("mdm -> pwr rows:", m2p.count())

---
## Stage 8 — Deposit panel: unpivot and profile

The deposit table is **wide** — one row per `cust_pwr_id`, months as columns,
**monthly average balance**. Unpivoted to a long panel here.

The null pattern in the `bal_*` series carries information that no other column
holds. Leading nulls are pre-tenure. **Trailing nulls are a candidate departure
label** — a customer whose balance series simply stops. That is closer to an
actual departure than any threshold on a declining balance, and it is available
today, so it is profiled here rather than pinned.

In [ ]:
dep_w = spark.table(CONFIG["DEP_TABLE"])
bal_cols = sorted([c for c in dep_w.columns if re.match(rf"{CONFIG['BAL_PREFIX']}\d{{4}}_\d{{2}}$", c)])
print(f"{len(bal_cols)} balance columns: {bal_cols[0]} .. {bal_cols[-1]}")

pairs = ", ".join([f"'{c[len(CONFIG['BAL_PREFIX']):].replace('_','-')}', cast({c} as double)"
                   for c in bal_cols])
dep = (dep_w.select(F.col(DEP["cust"]).alias("cust_pwr_id"),
                    F.col(DEP["rltn"]).alias("rltn_pwr_id"),
                    F.expr(f"stack({len(bal_cols)}, {pairs}) as (month, balance)"))
            .cache())

save(dep_w.agg(
        F.count(F.lit(1)).alias("n_rows"),
        F.countDistinct(DEP["cust"]).alias("n_customers"),
        F.countDistinct(DEP["rltn"]).alias("n_relationships"),
        F.mean(F.col(DEP["n_acct"]).cast("double")).alias("mean_accounts"),
        F.mean(F.col(DEP["n_closed"]).cast("double")).alias("mean_closed_accounts"),
        F.mean((F.col(DEP["n_closed"]) > 0).cast("double")).alias("share_any_closed"),
     ).toPandas(),
     "08_deposit_shape", "n_rows == n_customers confirms one row per customer")

In [ ]:
# Per-month non-null coverage. A ramp at either end is a panel artefact and will
# read as a level shift in every feature built on it.
save(dep.groupBy("month").agg(
        F.count(F.lit(1)).alias("n_customers"),
        F.mean(F.col("balance").isNotNull().cast("double")).alias("share_non_null"),
        F.expr("percentile_approx(balance, 0.5)").alias("p50_balance"),
        F.sum("balance").alias("total_balance")).orderBy("month").toPandas(),
     "08_deposit_month_coverage")

In [ ]:
# Null-pattern shapes. This is where a real departure label may already exist.
w = W.partitionBy("cust_pwr_id").orderBy("month")
pat = (dep
    .withColumn("has_bal", F.col("balance").isNotNull() & (F.col("balance") != 0))
    .groupBy("cust_pwr_id")
    .agg(F.sum(F.col("has_bal").cast("int")).alias("n_months_active"),
         F.min(F.when(F.col("has_bal"), F.col("month"))).alias("first_active"),
         F.max(F.when(F.col("has_bal"), F.col("month"))).alias("last_active"),
         F.max("month").alias("panel_end"))
    .withColumn("shape",
        F.when(F.col("n_months_active") == 0, "never_active")
         .when(F.col("last_active") == F.col("panel_end"), "active_at_end")
         .otherwise("STOPPED_BEFORE_END"))
    .cache())

save(pat.groupBy("shape").count().toPandas(), "08_balance_series_shapes",
     "*** STOPPED_BEFORE_END is a candidate departure label available TODAY ***")

save(pat.filter("shape='STOPPED_BEFORE_END'")
        .groupBy("last_active").count().orderBy("last_active").toPandas(),
     "08_stop_month_distribution",
     "a flat distribution is real churn; a spike at one month is a data artefact")
R["n_stopped_before_end"] = int(pat.filter("shape='STOPPED_BEFORE_END'").count())

In [ ]:
# The rule's own intermediates are already in the table. Reproduce pct_change from
# the unpivoted series and check it agrees — if it does not, prior_avg/recent_avg
# are computed on a different window or a different balance definition than the
# columns, and the panel cannot be used to re-derive the rule.
chk = (dep_w.select(F.col(DEP["cust"]).alias("cust_pwr_id"),
                    F.col(DEP["latest"]).alias("latest_month"),
                    F.col(DEP["prior_avg"]).cast("double").alias("prior_avg"),
                    F.col(DEP["recent_avg"]).cast("double").alias("recent_avg"),
                    F.col(DEP["pct_change"]).cast("double").alias("pct_change"))
       .withColumn("recomputed", (F.col("recent_avg") - F.col("prior_avg")) / F.col("prior_avg"))
       .withColumn("agrees", F.abs(F.col("recomputed") - F.col("pct_change")) < 0.001))
save(chk.agg(F.mean(F.col("agrees").cast("double")).alias("share_agreeing"),
             F.countDistinct("latest_month").alias("n_distinct_latest_month")).toPandas(),
     "08_rule_intermediates_check",
     "n_distinct_latest_month > 1 means latest_month varies per customer — check before pooling")

---
## Stage 9 — Coverage: deposit book ↔ staging

**The number that replaces the ~12% figure.** The prior verdict — graph as a
coincident rather than leading indicator — was explicitly conditional on that
coverage. Deposit-anchored extraction should push it near-total. If it does not,
the reason has to be found before anything is built on top.

In [ ]:
dep_custs = dep_w.select(F.col(DEP["cust"]).alias("cust_pwr_id")).distinct()

txn_custs = (txn.select(F.explode(F.array(F.col(T["pays_id"]), F.col(T["recv_id"]))).alias("mdm_id"))
                .filter(present("mdm_id")).distinct()
                .join(m2p, "mdm_id").select("cust_pwr_id").distinct())

cov = (dep_custs.join(txn_custs.withColumn("_seen", F.lit(1)), "cust_pwr_id", "left")
    .agg(F.count(F.lit(1)).alias("n_deposit_customers"),
         F.sum(F.coalesce("_seen", F.lit(0))).alias("n_with_transactions")).toPandas())
cov["coverage"] = cov["n_with_transactions"] / cov["n_deposit_customers"]
save(cov, "09_deposit_txn_coverage", "*** replaces the ~12% figure ***")
R["deposit_txn_coverage"] = float(cov["coverage"].iloc[0])

# Coverage split by OUTBOUND specifically — Group A needs the outbound side, and a
# customer visible only as a payee cannot support any of it.
out_custs = (txn.filter("topology='OUTBOUND'")
                .select(F.col(T["pays_id"]).alias("mdm_id")).distinct()
                .join(m2p, "mdm_id").select("cust_pwr_id").distinct())
cov2 = (dep_custs.join(out_custs.withColumn("_out", F.lit(1)), "cust_pwr_id", "left")
    .agg(F.count(F.lit(1)).alias("n_deposit_customers"),
         F.sum(F.coalesce("_out", F.lit(0))).alias("n_with_outbound")).toPandas())
cov2["coverage"] = cov2["n_with_outbound"] / cov2["n_deposit_customers"]
save(cov2, "09_deposit_outbound_coverage", "the population Group A can actually be tested on")

# Dtype trap — the 2026-07 incident. Both sides must be strings.
print("dep cust dtype:", dict(dep_w.dtypes)[DEP["cust"]],
      "| cust dim pwr dtype:", dict(cust.dtypes)[CUST["pwr_id"]],
      "| txn pays dtype:", dict(txn.dtypes)[T["pays_id"]])

---
## Stage 10 — Does staging explain balance movement?

**Exact reconciliation is impossible here** — the balance is a *monthly average*,
so its month-over-month delta is a smoothed function of within-month flow, not
the sum of it. Any residual is therefore partly the averaging and partly missing
data, and the two cannot be separated.

What is measurable, and what actually matters:

1. **Explained variance** of balance delta on net flow. High R² means one ledger:
   deposit and payment features are two views of one table, and the only new axis
   the graph contributes is **counterparty identity**, not amount or timing. That
   is still a real contribution — but it changes the hypothesis and predicts which
   metrics can add lift.
2. **Customer-months with balance movement but zero staging rows.** These are
   money movements staging does not carry, and that share is a hard ceiling on
   anything transaction-derived.

In [ ]:
flow = (txn.withColumn("month", F.substring(T["date"], 1, 7))
    .withColumn("mdm_id", F.coalesce(F.col(T["pays_id"]), F.col(T["recv_id"])))
    .withColumn("signed",
        F.when(F.col("topology") == "OUTBOUND", -F.col(T["amount"]).cast("double"))
         .when(F.col("topology") == "INBOUND",   F.col(T["amount"]).cast("double"))
         .otherwise(F.lit(0.0)))          # INTERNAL_C2C nets out at customer level
    .filter(present("mdm_id"))
    .join(m2p, "mdm_id")
    .groupBy("cust_pwr_id", "month")
    .agg(F.sum("signed").alias("net_flow"),
         F.sum(F.when(F.col("topology") == "OUTBOUND", F.col(T["amount"]).cast("double")).otherwise(0.0)).alias("gross_out"),
         F.sum(F.when(F.col("topology") == "INBOUND",  F.col(T["amount"]).cast("double")).otherwise(0.0)).alias("gross_in"),
         F.count(F.lit(1)).alias("n_txn")))

wd = W.partitionBy("cust_pwr_id").orderBy("month")
panel = (dep.withColumn("bal_prev", F.lag("balance").over(wd))
    .withColumn("bal_delta", F.col("balance") - F.col("bal_prev"))
    .filter(F.col("bal_prev").isNotNull() & F.col("balance").isNotNull())
    .join(flow, ["cust_pwr_id", "month"], "left")
    .fillna({"net_flow": 0.0, "gross_out": 0.0, "gross_in": 0.0, "n_txn": 0})
    .cache())

save(panel.agg(
        F.count(F.lit(1)).alias("n_customer_months"),
        F.corr("bal_delta", "net_flow").alias("corr_delta_netflow"),
        F.corr("balance", "gross_out").alias("corr_level_grossout"),
     ).toPandas(),
     "10_flow_vs_balance",
     "*** corr near 1 => ONE ledger; identity is the new axis, not amount ***")
R["corr_delta_netflow"] = float(panel.agg(F.corr("bal_delta", "net_flow")).first()[0] or 0)

In [ ]:
save(panel.withColumn("bucket",
        F.when((F.col("n_txn") == 0) & (F.abs(F.col("bal_delta")) > 1000), "NO_TXN_BUT_BALANCE_MOVED")
         .when(F.col("n_txn") == 0, "no_txn_flat_balance")
         .otherwise("has_txn"))
     .groupBy("bucket").agg(
        F.count(F.lit(1)).alias("n"),
        F.expr("percentile_approx(abs(bal_delta), 0.5)").alias("p50_abs_delta")).toPandas(),
     "10_unexplained_movement",
     "*** NO_TXN_BUT_BALANCE_MOVED is a hard ceiling on transaction-derived features ***")

---
## Stage 11 — Account activity as a closure proxy

`latest_num_closed_accounts` is current-state only, so it cannot date a closure
and cannot serve as a time-varying label — using it as one would leak the outcome
into every prior month. **A monthly version is the pinned ask to the deposit team.**

Meanwhile `pnc_dep_acct_pays` / `pnc_dep_acct_receives` give a per-month count of
accounts that *transacted*. An account that goes permanently silent is a
behavioural closure, dated, and available today. This stage measures whether that
proxy has enough signal to stand in.

In [ ]:
acct_act = (txn.withColumn("month", F.substring(T["date"], 1, 7))
    .withColumn("mdm_id", F.coalesce(F.col(T["pays_id"]), F.col(T["recv_id"])))
    .withColumn("acct", F.coalesce(F.col(T["pays_acct"]), F.col(T["recv_acct"])))
    .filter(present("mdm_id") & present("acct"))
    .join(m2p, "mdm_id")
    .groupBy("cust_pwr_id", "month").agg(F.countDistinct("acct").alias("n_active_accts")))

save(acct_act.groupBy("month").agg(
        F.count(F.lit(1)).alias("n_customers"),
        F.mean("n_active_accts").alias("mean_active_accts")).orderBy("month").toPandas(),
     "11_active_accounts_by_month")

# Does the transaction-derived account count track the deposit table's stated count?
cmp_ = (acct_act.groupBy("cust_pwr_id").agg(F.max("n_active_accts").alias("txn_max_accts"))
    .join(dep_w.select(F.col(DEP["cust"]).alias("cust_pwr_id"),
                       F.col(DEP["n_acct"]).cast("int").alias("dep_n_accts"),
                       F.col(DEP["n_closed"]).cast("int").alias("dep_n_closed")),
          "cust_pwr_id"))
save(cmp_.agg(
        F.corr("txn_max_accts", "dep_n_accts").alias("corr"),
        F.mean((F.col("txn_max_accts") == F.col("dep_n_accts")).cast("double")).alias("share_exact"),
        F.mean(F.col("txn_max_accts").cast("double")).alias("mean_txn"),
        F.mean(F.col("dep_n_accts").cast("double")).alias("mean_dep")).toPandas(),
     "11_account_count_agreement",
     "*** if these agree, account silence is a usable dated closure proxy ***")

# Do customers with closed accounts show account count DECLINING in the txn data?
# This is the validation of the proxy against the one closure fact that exists.
decline = (acct_act
    .withColumn("rn_first", F.first("n_active_accts").over(W.partitionBy("cust_pwr_id").orderBy("month")))
    .withColumn("rn_last", F.last("n_active_accts").over(
        W.partitionBy("cust_pwr_id").orderBy("month").rowsBetween(0, W.unboundedFollowing)))
    .select("cust_pwr_id", "rn_first", "rn_last").distinct()
    .withColumn("acct_declined", F.col("rn_last") < F.col("rn_first"))
    .join(dep_w.select(F.col(DEP["cust"]).alias("cust_pwr_id"),
                       (F.col(DEP["n_closed"]) > 0).alias("has_closed")), "cust_pwr_id"))
save(decline.groupBy("has_closed", "acct_declined").count().toPandas(),
     "11_closure_proxy_validation",
     "*** the confusion matrix for the behavioural closure proxy ***")

---
## Stage 12 — Same-name outflow: base rate and false-positive estimate

The brief calls `same_name_outflow_flag` its priority signal. Two things have to
be measured before it can be used:

1. **The base rate.** Most corporates permanently maintain multiple bank
   relationships. If a large share of the book always shows same-name outflow,
   the flag as specified fires constantly and carries nothing. **The signal is a
   NEW destination or a step change in share — a delta, not a level.**
2. **The false-positive rate of exact name matching.** Estimable without any
   labelling: `INTERNAL_C2C` rows where the two MDM ids DIFFER but the normalised
   names match are, by construction, name-match false positives — two distinct
   legal entities sharing a name. That rate transfers to the off-us case, with
   the caveat that off-us names come from rail fields and are dirtier.

In [ ]:
fp = (txn.filter("topology='INTERNAL_C2C' AND NOT is_self_loop")
    .withColumn("name_match", norm_name(F.col(T["pays_name"])) == norm_name(F.col(T["recv_name"])))
    .agg(F.count(F.lit(1)).alias("n_distinct_entity_pairs"),
         F.sum(F.col("name_match").cast("int")).alias("n_false_name_matches"),
         F.mean(F.col("name_match").cast("double")).alias("false_positive_rate")).toPandas())
save(fp, "12_name_match_false_positives",
     "*** distinct entities whose names collide — the FP rate exact matching inherits ***")

In [ ]:
same_name = (txn.filter("topology='OUTBOUND'")
    .filter(present(T["cpty_name"]) & present(T["pays_name"]))
    .withColumn("month", F.substring(T["date"], 1, 7))
    .withColumn("same_name", norm_name(F.col(T["cpty_name"])) == norm_name(F.col(T["pays_name"])))
    .cache())

base = (same_name.groupBy("month")
    .agg(F.countDistinct(F.when(F.col("same_name"), F.col(T["pays_id"]))).alias("n_cust_same_name"),
         F.countDistinct(T["pays_id"]).alias("n_cust_total"),
         (F.sum(F.when(F.col("same_name"), F.col(T["amount"]).cast("double")).otherwise(0.0))
          / F.sum(F.col(T["amount"]).cast("double"))).alias("dollar_share"))
    .withColumn("base_rate", F.col("n_cust_same_name") / F.col("n_cust_total"))
    .orderBy("month").toPandas())
save(base, "12_same_name_base_rate",
     "*** high and flat => measure the DELTA, not the LEVEL ***")
R["same_name_base_rate"] = float(base["base_rate"].mean()) if len(base) else None

# Which banks receive same-name outflow. This is the wallet-share picture the
# counterparty work was meant to unlock, available now for this population.
save(same_name.filter("same_name").groupBy(T["cpty_fi"])
        .agg(F.countDistinct(T["pays_id"]).alias("n_customers"),
             F.sum(F.col(T["amount"]).cast("double")).alias("dollars"))
        .orderBy(F.desc("n_customers")).limit(40).toPandas(),
     "12_same_name_destination_banks", "where customers hold their other accounts")

---
## Stage 13 — Episode funnel and power

The 30% rule applied to the unpivoted panel, then the brief's §6.2 exclusions,
then graph visibility. **This decides whether Steps 3–6 are worth running.**
Twenty-five metrics against matched controls at two horizons needs thousands of
episodes, not hundreds.

In [ ]:
w13 = W.partitionBy("cust_pwr_id").orderBy("month")
ep = (dep.filter(F.col("balance").isNotNull())
    .withColumn("avg3",  F.avg("balance").over(w13.rowsBetween(-2, 0)))
    .withColumn("avg6p", F.avg("balance").over(w13.rowsBetween(-8, -3)))
    .withColumn("n_hist", F.count("balance").over(w13.rowsBetween(-11, 0)))
    .withColumn("fwd3",  F.avg("balance").over(w13.rowsBetween(1, 3)))
    .withColumn("flag", F.col("avg3") <= (1 - CONFIG["CURRENT_RULE_DROP"]) * F.col("avg6p")))

funnel, prev = [], None
for label, df_ in [
    ("1_raw_30pct_rule",        ep),
    ("2_plus_12mo_history",     ep.filter(F.col("n_hist") >= 12)),
    ("3_plus_balance_floor",    ep.filter((F.col("n_hist") >= 12) & (F.col("avg6p") >= CONFIG["BALANCE_FLOOR"]))),
    ("4_plus_persistence",      ep.filter((F.col("n_hist") >= 12) & (F.col("avg6p") >= CONFIG["BALANCE_FLOOR"])
                                          & (F.col("fwd3") <= 1.1 * F.col("avg3")))),
]:
    n = df_.filter("flag").select("cust_pwr_id").distinct().count()
    funnel.append({"step": label, "n_customers": n})
save(pd.DataFrame(funnel), "13_episode_funnel", "*** the last row is the real sample size ***")

In [ ]:
survivors = (ep.filter((F.col("n_hist") >= 12) & (F.col("avg6p") >= CONFIG["BALANCE_FLOOR"])
                       & (F.col("fwd3") <= 1.1 * F.col("avg3")) & F.col("flag"))
               .select("cust_pwr_id").distinct())

gv = (survivors
    .join(txn_custs.withColumn("_any", F.lit(1)), "cust_pwr_id", "left")
    .join(out_custs.withColumn("_out", F.lit(1)), "cust_pwr_id", "left")
    .agg(F.count(F.lit(1)).alias("n_episodes"),
         F.sum(F.coalesce("_any", F.lit(0))).alias("n_any_txn"),
         F.sum(F.coalesce("_out", F.lit(0))).alias("n_outbound_visible")).toPandas())
gv["coverage_outbound"] = gv["n_outbound_visible"] / gv["n_episodes"]
save(gv, "13_episodes_graph_visible",
     "*** n_outbound_visible is the population Group A can be tested on ***")
R["episodes_outbound_visible"] = int(gv["n_outbound_visible"].iloc[0])

---
## Stage 14 — Candidate labels

No status field exists. Three candidate targets are compared here so the choice
is made on evidence rather than by default:

| Label | Source | Circular? | Dated? |
|---|---|---|---|
| `balance_30pct` | the current rule | **yes** — deposit-defined | yes |
| `series_stopped` | trailing nulls in `bal_*` | no | yes |
| `txn_silence` | N consecutive months with no staging rows | **no** — independent of deposits | yes |

`txn_silence` is the one that breaks the circularity. A deposit model predicting a
deposit-derived event wins by construction; a behaviour-defined target is a fair
test of whether payment structure leads. The agreement matrix below shows how much
these three actually disagree — if they are near-identical the choice is moot, and
if they are not, the choice is the most consequential decision in the study.

In [ ]:
active = (txn.withColumn("month", F.substring(T["date"], 1, 7))
    .withColumn("mdm_id", F.coalesce(F.col(T["pays_id"]), F.col(T["recv_id"])))
    .filter(present("mdm_id")).join(m2p, "mdm_id")
    .groupBy("cust_pwr_id", "month").agg(F.count(F.lit(1)).alias("n_txn")))

grid = dep.select("cust_pwr_id", "month").join(active, ["cust_pwr_id", "month"], "left") \
          .fillna({"n_txn": 0})
w14 = W.partitionBy("cust_pwr_id").orderBy("month")
silence = (grid
    .withColumn("fwd_activity", F.sum("n_txn").over(w14.rowsBetween(0, CONFIG["SILENCE_MONTHS"] - 1)))
    .withColumn("silent", F.col("fwd_activity") == 0)
    .groupBy("cust_pwr_id").agg(F.max(F.col("silent").cast("int")).alias("txn_silence")))

labels = (dep_custs
    .join(silence, "cust_pwr_id", "left")
    .join(pat.select("cust_pwr_id", F.when(F.col("shape") == "STOPPED_BEFORE_END", 1).otherwise(0)
                     .alias("series_stopped")), "cust_pwr_id", "left")
    .join(survivors.withColumn("balance_30pct", F.lit(1)), "cust_pwr_id", "left")
    .fillna({"txn_silence": 0, "series_stopped": 0, "balance_30pct": 0}).cache())

save(labels.groupBy("balance_30pct", "series_stopped", "txn_silence").count()
           .orderBy(F.desc("count")).toPandas(),
     "14_label_agreement",
     "*** how much the three candidate targets disagree ***")
save(labels.agg(F.mean("balance_30pct").alias("rate_balance_30pct"),
                F.mean("series_stopped").alias("rate_series_stopped"),
                F.mean(F.col("txn_silence").cast("double")).alias("rate_txn_silence")).toPandas(),
     "14_label_prevalence", "prevalence drives every downstream power calculation")

---
## Stage 15 — Summary

In [ ]:
save(pd.DataFrame(list(R.items()), columns=["metric", "value"]), "15_SUMMARY")

print("""
READING THE SUMMARY
-------------------
outbound_dollar_share          the half of the payment picture the graph has never had.
cpty_name_dollar_coverage      >0.70 Group A live as specified · 0.40-0.70 live on a biased
                               subset, always quote the rail split · <0.40 rail-specific metric.
cpty_key_dollar_coverage       with account_derived provenance dominant, counterparty nodes are
                               stable and Group B relationship-dissolution metrics are trustworthy.
                               If name_derived dominates, a spelling change reads as a lost
                               relationship and Group B needs a stability caveat.
corr_delta_netflow             >0.8 ONE ledger. Reframe: counterparty IDENTITY is the new axis,
                               not amount or timing. <0.4 two systems; independent-sources framing
                               holds. Note the monthly-average balance caps this regardless.
deposit_txn_coverage           replaces the ~12% figure that made the prior verdict conditional.
same_name_base_rate            high and flat => the brief's flag fires constantly; use the delta.
episodes_outbound_visible      <300 underpowered, ship Steps 1-2 and stop · 300-1000 descriptive
                               separation curves only · >1000 the brief's plan runs as written.

PINNED FOR THE DEPOSIT TEAM
    monthly closed-account count, or per-account open/close dates. Converts this from
    predicting a balance-derived event to predicting an actual departure.
""")